# Session 13B — Face Recognition Pipeline
**Phase 3 | Week 9 | Computer Vision & AI Course**

---

## What We'll Cover

| Step | Topic |
|---|---|
| 1 | Face Detection — finding faces in images |
| 2 | Face Alignment — normalising face orientation |
| 3 | Embedding — turning a face into a feature vector |
| 4 | Matching — comparing embeddings to recognise identity |
| 5 | Full Pipeline — putting it all together |
| 6 | DeepFace — high-level library overview |

**Estimated time:** ~45 minutes

> 💡 **Goal:** Understand each stage of a production face recognition system — from raw image to identity match — and implement a simple face ID pipeline from scratch.

---
## 1. How Face Recognition Works

Face recognition is **not** a single model — it's a **pipeline** of four distinct stages:

```
Input Image
     │
     ▼
┌─────────────┐
│  DETECTION  │  ← Find where faces are (bounding boxes)
└─────────────┘
     │
     ▼
┌─────────────┐
│  ALIGNMENT  │  ← Rotate & crop so eyes are always horizontal
└─────────────┘
     │
     ▼
┌─────────────┐
│  EMBEDDING  │  ← Deep neural net → 512-dim face vector
└─────────────┘
     │
     ▼
┌─────────────┐
│   MATCHING  │  ← Cosine similarity vs known faces database
└─────────────┘
     │
     ▼
  "Person: Alice"  or  "Unknown"
```

---

### Stage 1: Face Detection
We need to find *where* faces are before we can recognise them.
- **MTCNN** — accurate, returns landmarks (eyes, nose, mouth)
- **RetinaFace** — state-of-the-art, very fast
- **OpenCV Haar Cascade** — classic, fast but less accurate
- **YOLOv8-face** — real-time multi-face

### Stage 2: Face Alignment
The same person looks very different if their head is tilted. Alignment:
- Uses the **eye landmark positions** detected in Stage 1
- Applies a rotation so that both eyes are at the **same height**
- Crops and resizes to a standard size (e.g., 112×112)

This makes recognition *much* more robust.

### Stage 3: Face Embedding (ArcFace)
A deep CNN converts the aligned face crop into a **compact vector** (embedding):
- Typically **512 dimensions**
- Faces of the *same person* are **close** in embedding space
- Faces of *different people* are **far apart**

**ArcFace** is the current state-of-the-art embedding model. It uses an **Additive Angular Margin Loss** during training to make same-person embeddings tighter and different-person embeddings more separated.

### Stage 4: Matching
Compare the query embedding to a **database of known embeddings** using **cosine similarity**:

```
cosine_similarity(A, B) = (A · B) / (|A| × |B|)

Result is between -1 and 1:
  ~1.0 = very similar (same person)
  ~0.0 = unrelated (different person)
```

If the best match score exceeds a **threshold** (typically 0.6–0.7), we declare a match.

---
## 2. Setup & Installations

In [ ]:
# Install required libraries

# face_recognition: high-level face recognition library (wraps dlib)
!pip install face_recognition opencv-python matplotlib numpy Pillow -q

# deepface: comprehensive face analysis framework (detection, recognition, age, emotion)
!pip install deepface -q

# insightface: state-of-the-art ArcFace implementation
!pip install insightface onnxruntime -q

# mtcnn: for face detection with landmarks
!pip install mtcnn tensorflow -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
import urllib.request

# Helper: display images
def show_image(img, title='', figsize=(8, 6)):
    if isinstance(img, np.ndarray) and len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_images_row(images, titles, figsize=(16, 5)):
    """Display multiple images side by side."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if isinstance(img, np.ndarray) and len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✅ Imports successful!")

---
## 3. Download Sample Face Images

We'll use the **Labeled Faces in the Wild (LFW)** sample images for demonstration. These are free public-domain face images commonly used for benchmarking face recognition systems.

In [ ]:
# ── Create synthetic face-like test images ────────────────────────────────────
# (Since downloading real faces requires network access,
#  we generate simple placeholder images for structural testing.)

# Create a faces/ directory
os.makedirs('faces/alice', exist_ok=True)
os.makedirs('faces/bob', exist_ok=True)
os.makedirs('faces/query', exist_ok=True)

def create_synthetic_face(filename, eye_y=60, eye_spacing=40, 
                           skin_color=(210, 180, 140), hair_color=(80, 50, 30)):
    """
    Create a very simple synthetic face image for structural testing.
    In a real session, use actual face photos from a dataset or webcam.
    """
    img = np.ones((112, 112, 3), dtype=np.uint8) * 200  # light grey bg
    
    # Head (oval)
    cv2.ellipse(img, (56, 65), (38, 45), 0, 0, 360, skin_color, -1)
    
    # Hair
    cv2.ellipse(img, (56, 30), (38, 25), 0, 180, 360, hair_color, -1)
    
    # Eyes
    cv2.circle(img, (56 - eye_spacing//2, eye_y), 6, (50, 50, 50), -1)
    cv2.circle(img, (56 + eye_spacing//2, eye_y), 6, (50, 50, 50), -1)
    # Eye whites
    cv2.circle(img, (56 - eye_spacing//2, eye_y), 3, (255, 255, 255), -1)
    cv2.circle(img, (56 + eye_spacing//2, eye_y), 3, (255, 255, 255), -1)
    
    # Nose
    cv2.circle(img, (56, 80), 4, (180, 140, 110), -1)
    
    # Mouth
    cv2.ellipse(img, (56, 92), (12, 6), 0, 0, 180, (180, 80, 80), -1)
    
    cv2.imwrite(filename, img)
    return img


# Create faces for 2 identities with slight variations
# Alice: lighter skin, wider eye spacing
alice_1 = create_synthetic_face('faces/alice/alice_1.jpg', eye_y=60, eye_spacing=40,
                                  skin_color=(220, 190, 155))
alice_2 = create_synthetic_face('faces/alice/alice_2.jpg', eye_y=58, eye_spacing=42,
                                  skin_color=(215, 185, 150))

# Bob: darker skin, closer eye spacing
bob_1 = create_synthetic_face('faces/bob/bob_1.jpg', eye_y=62, eye_spacing=34,
                                skin_color=(160, 120, 90), hair_color=(30, 20, 20))
bob_2 = create_synthetic_face('faces/bob/bob_2.jpg', eye_y=60, eye_spacing=36,
                                skin_color=(155, 115, 85), hair_color=(25, 15, 15))

# Query faces (will try to match against known faces)
query_alice = create_synthetic_face('faces/query/query_alice.jpg', eye_y=61, eye_spacing=41,
                                     skin_color=(218, 188, 153))
query_bob   = create_synthetic_face('faces/query/query_bob.jpg', eye_y=63, eye_spacing=35,
                                     skin_color=(158, 118, 88), hair_color=(28, 18, 18))

# Show all faces
show_images_row(
    [alice_1, alice_2, bob_1, bob_2],
    ['Alice (ref 1)', 'Alice (ref 2)', 'Bob (ref 1)', 'Bob (ref 2)']
)
show_images_row(
    [query_alice, query_bob],
    ['Query: "Who is this?" (Alice)', 'Query: "Who is this?" (Bob)']
)
print("\n⚠️  These are synthetic faces for structural demonstration.")
print("    For a real demo, replace with actual face photos in the faces/ directory.")
print("    Good sources: your webcam, LFW dataset, or any Creative Commons images.")

---
## 4. Stage 1 — Face Detection

We'll use **MTCNN** (Multi-task Cascaded Convolutional Networks), which:
- Detects faces at multiple scales
- Returns **5 facial landmarks**: left eye, right eye, nose, left mouth corner, right mouth corner
- These landmarks are essential for alignment

In [ ]:
# ── Face Detection with MTCNN ─────────────────────────────────────────────────

from mtcnn import MTCNN

# Initialise detector
detector = MTCNN()

def detect_faces_mtcnn(image_path):
    """
    Detect all faces in an image using MTCNN.
    
    Returns:
        img_rgb : the image as RGB numpy array
        faces   : list of dicts, each with:
                  - 'box': [x, y, width, height]
                  - 'confidence': float
                  - 'keypoints': {'left_eye', 'right_eye', 'nose', ...}
    """
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    faces = detector.detect_faces(img_rgb)
    return img_rgb, faces


# Test on Alice's first photo
img_rgb, faces = detect_faces_mtcnn('faces/alice/alice_1.jpg')

print(f"Faces detected: {len(faces)}")
for i, face in enumerate(faces):
    print(f"\nFace {i+1}:")
    print(f"  Bounding box: {face['box']}  (x, y, width, height)")
    print(f"  Confidence:   {face['confidence']:.3f}")
    print(f"  Keypoints:    {face['keypoints']}")

In [ ]:
# ── Visualise Detection Results ───────────────────────────────────────────────

def draw_face_detections(img_rgb, faces):
    """
    Draw bounding boxes and facial landmarks on an image.
    """
    img = img_rgb.copy()
    
    for face in faces:
        x, y, w, h = face['box']
        conf = face['confidence']
        keypoints = face['keypoints']
        
        # Draw bounding box
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(img, f"{conf:.2f}", (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
        # Draw landmarks
        landmark_colors = {
            'left_eye':  (255, 0, 0),    # red
            'right_eye': (0, 0, 255),    # blue
            'nose':      (0, 255, 255),  # cyan
            'mouth_left': (255, 255, 0), # yellow
            'mouth_right': (255, 0, 255) # magenta
        }
        for name, pt in keypoints.items():
            color = landmark_colors.get(name, (255, 255, 255))
            cv2.circle(img, pt, 3, color, -1)
    
    return img


annotated = draw_face_detections(img_rgb, faces)
show_image(annotated, title="MTCNN Detection — Bounding Box + 5 Landmarks")

print("Landmark colour guide:")
print("  🔴 Left eye  |  🔵 Right eye  |  🟦 Nose  |  🟡 Mouth left  |  🟣 Mouth right")

---
## 5. Stage 2 — Face Alignment

Alignment normalises the face so that:
- Both eyes are at the same vertical position
- The face is centred and cropped to a fixed size

Without alignment, a tilted face will produce a very different embedding, making recognition unreliable.

In [ ]:
# ── Face Alignment ────────────────────────────────────────────────────────────

def align_face(img_rgb, face_info, output_size=(112, 112)):
    """
    Align a detected face so that eyes are horizontal.
    
    Steps:
    1. Find the angle between the two eyes
    2. Rotate the image to make eyes horizontal
    3. Crop and resize to output_size
    
    Parameters:
        img_rgb   : full image as RGB numpy array
        face_info : single face dict from MTCNN (with 'keypoints' and 'box')
        output_size : (width, height) of aligned face crop
    
    Returns:
        aligned_face : RGB numpy array of shape output_size
    """
    keypoints = face_info['keypoints']
    x, y, w, h = face_info['box']
    
    left_eye  = np.array(keypoints['left_eye'],  dtype=np.float32)
    right_eye = np.array(keypoints['right_eye'], dtype=np.float32)
    
    # Step 1: Calculate angle between eyes
    # (positive angle = right eye is higher than left)
    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))  # angle in degrees
    
    # Step 2: Get the midpoint between eyes (rotation centre)
    eye_centre = ((left_eye + right_eye) / 2).astype(int)
    
    # Step 3: Build rotation matrix and rotate the image
    M = cv2.getRotationMatrix2D(tuple(eye_centre), angle, scale=1.0)
    rotated = cv2.warpAffine(img_rgb, M, (img_rgb.shape[1], img_rgb.shape[0]))
    
    # Step 4: Crop the face region (add padding around bounding box)
    pad = int(max(w, h) * 0.2)  # 20% padding
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(img_rgb.shape[1], x + w + pad)
    y2 = min(img_rgb.shape[0], y + h + pad)
    
    face_crop = rotated[y1:y2, x1:x2]
    
    # Step 5: Resize to standard output size
    if face_crop.size == 0:
        # Fallback: just crop and resize without rotation
        face_crop = img_rgb[y:y+h, x:x+w]
    
    aligned = cv2.resize(face_crop, output_size)
    
    return aligned, angle


# Align Alice's face
if faces:
    aligned_face, angle = align_face(img_rgb, faces[0])
    
    # Compare: raw crop vs aligned
    x, y, w, h = faces[0]['box']
    raw_crop = img_rgb[y:y+h, x:x+w]
    raw_crop_resized = cv2.resize(raw_crop, (112, 112))
    
    show_images_row(
        [raw_crop_resized, aligned_face],
        [f'Raw Crop', f'Aligned (rotation: {angle:.1f}°)']
    )
    print(f"Eye alignment angle: {angle:.2f}°")
    print(f"Output face size: {aligned_face.shape}")
else:
    print("No faces detected to align.")

---
## 6. Stage 3 — Face Embedding

### What is an embedding?

A **face embedding** is a fixed-size vector (e.g., 512 numbers) that represents the unique facial characteristics of a person. A deep neural network learns to map faces to this embedding space such that:

```
Same person  →  embeddings are CLOSE  (cosine similarity ≈ 1)
Diff person  →  embeddings are FAR    (cosine similarity ≈ 0 or negative)
```

### ArcFace loss (intuition)

Traditional classification: "is this person A, B, or C?" — doesn't generalise to new identities.

ArcFace trains with **metric learning** + **angular margin**:
- Projects embeddings onto a unit hypersphere
- Adds a margin (angular penalty) to force same-class embeddings to be *tighter*
- Result: embeddings generalise to **unseen identities** at inference time

In [ ]:
# ── Face Embeddings with face_recognition library ─────────────────────────────
# face_recognition wraps dlib's ResNet-based face encoder
# It produces 128-dim embeddings

import face_recognition

def get_face_embedding(image_path):
    """
    Compute the 128-dim face embedding for the first detected face in an image.
    
    Returns:
        embedding: np.array of shape (128,), or None if no face found
    """
    # Load image
    img = face_recognition.load_image_file(image_path)
    
    # Detect face locations first
    locations = face_recognition.face_locations(img)
    
    if not locations:
        return None
    
    # Compute embeddings (one per detected face)
    # model='large' is more accurate, 'small' is faster
    embeddings = face_recognition.face_encodings(img, locations, model='large')
    
    return embeddings[0] if embeddings else None


# Get embeddings for Alice and Bob's reference images
alice_emb_1 = get_face_embedding('faces/alice/alice_1.jpg')
alice_emb_2 = get_face_embedding('faces/alice/alice_2.jpg')
bob_emb_1   = get_face_embedding('faces/bob/bob_1.jpg')
bob_emb_2   = get_face_embedding('faces/bob/bob_2.jpg')

# Check what an embedding looks like
if alice_emb_1 is not None:
    print(f"Embedding shape: {alice_emb_1.shape}")
    print(f"Embedding values (first 10): {alice_emb_1[:10].round(4)}")
    print(f"Embedding norm: {np.linalg.norm(alice_emb_1):.4f}  (should be ~1.0 for normalised embeddings)")
else:
    print("⚠️  No face detected. The synthetic test images may not be realistic enough.")
    print("    Please use real face photos for meaningful embeddings.")

---
## 7. Stage 4 — Face Matching

### Cosine Similarity

We measure how *similar* two face embeddings are using **cosine similarity**:

```
         A · B
cos(θ) = ─────────
         |A| × |B|
```

- **1.0** = identical vectors (same face)
- **0.5** = somewhat similar
- **0.0** = unrelated

### Euclidean Distance (alternative)
The `face_recognition` library uses **Euclidean distance** (lower = more similar):
- < 0.6 = same person (typical threshold)
- > 0.6 = different people

In [ ]:
# ── Similarity Functions ──────────────────────────────────────────────────────

def cosine_similarity(a, b):
    """
    Cosine similarity between two vectors.
    Returns a value between -1 and 1 (higher = more similar).
    """
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def euclidean_distance(a, b):
    """
    Euclidean distance between two vectors.
    Returns a positive value (lower = more similar).
    """
    return np.linalg.norm(a - b)


# Compare embeddings only if they exist
if alice_emb_1 is not None and alice_emb_2 is not None and bob_emb_1 is not None:
    
    # Compare Alice vs Alice (should be HIGH similarity)
    sim_alice_alice = cosine_similarity(alice_emb_1, alice_emb_2)
    dist_alice_alice = euclidean_distance(alice_emb_1, alice_emb_2)
    
    # Compare Alice vs Bob (should be LOW similarity)
    sim_alice_bob = cosine_similarity(alice_emb_1, bob_emb_1)
    dist_alice_bob = euclidean_distance(alice_emb_1, bob_emb_1)
    
    # Compare Bob vs Bob
    sim_bob_bob = cosine_similarity(bob_emb_1, bob_emb_2)
    dist_bob_bob = euclidean_distance(bob_emb_1, bob_emb_2)
    
    print("Similarity Results:")
    print("─" * 55)
    print(f"{'Comparison':<25} {'Cosine Sim':>12} {'Euclidean':>12}")
    print("─" * 55)
    print(f"{'Alice ↔ Alice (same)':<25} {sim_alice_alice:>12.4f} {dist_alice_alice:>12.4f}")
    print(f"{'Bob   ↔ Bob   (same)':<25} {sim_bob_bob:>12.4f} {dist_bob_bob:>12.4f}")
    print(f"{'Alice ↔ Bob   (diff)':<25} {sim_alice_bob:>12.4f} {dist_alice_bob:>12.4f}")
    print("─" * 55)
    print("\nExpected: same-person scores should be HIGHER (cosine) / LOWER (euclidean)")
else:
    print("Skipping similarity comparison — embeddings not available.")
    print("(Use real face images to see meaningful similarity scores.)")

In [ ]:
# ── Visualise Embeddings (if available) ──────────────────────────────────────

# Demonstrate the concept of embedding space with synthetic vectors
# (even if real face embeddings weren't extracted)

np.random.seed(42)

# Create synthetic embeddings to show the concept
# (in a real system these come from the neural network)
alice_base = np.random.randn(128)
alice_base /= np.linalg.norm(alice_base)

# Same person: small perturbation
alice_photo2 = alice_base + np.random.randn(128) * 0.05
alice_photo2 /= np.linalg.norm(alice_photo2)

# Different person: large difference
bob_base = np.random.randn(128)
bob_base /= np.linalg.norm(bob_base)

print("Synthetic embedding demonstration:")
print(f"  Alice photo 1 ↔ Alice photo 2 (SAME person): {cosine_similarity(alice_base, alice_photo2):.4f}")
print(f"  Alice photo 1 ↔ Bob           (DIFF person): {cosine_similarity(alice_base, bob_base):.4f}")

# Bar chart comparison
comparisons = {
    'Alice ↔ Alice\n(same)': cosine_similarity(alice_base, alice_photo2),
    'Alice ↔ Bob\n(diff)':   cosine_similarity(alice_base, bob_base),
}

colors = ['green' if v > 0.5 else 'red' for v in comparisons.values()]

plt.figure(figsize=(7, 4))
bars = plt.bar(comparisons.keys(), comparisons.values(), color=colors, alpha=0.8, edgecolor='white')
plt.axhline(y=0.6, color='orange', linestyle='--', label='Threshold (0.6)')
plt.ylim(0, 1.1)
plt.ylabel('Cosine Similarity', fontsize=12)
plt.title('Face Embedding Similarity (Synthetic Demo)', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print("\n✅ Values above the threshold (0.6) → same person")
print("   Values below the threshold      → different person")

---
## 8. Full Pipeline — Face Recognition System

Now let's combine all stages into a complete, reusable face recognition system.

In [ ]:
# ── FaceRecognizer: Complete Pipeline Class ───────────────────────────────────

class FaceRecognizer:
    """
    A complete face recognition pipeline:
      1. Detect faces (MTCNN)
      2. Compute embeddings (face_recognition / dlib)
      3. Match against a registered database
    
    Usage:
        recognizer = FaceRecognizer(threshold=0.6)
        recognizer.register('Alice', 'alice_1.jpg')
        recognizer.register('Alice', 'alice_2.jpg')
        result = recognizer.identify('query.jpg')
    """
    
    def __init__(self, threshold=0.6):
        """
        Parameters:
            threshold: Euclidean distance threshold.
                       Below this = same person, above = unknown.
                       (face_recognition library uses Euclidean distance)
        """
        self.threshold = threshold
        # Database: {name: [embedding1, embedding2, ...]}
        self.database = {}
    
    def register(self, name, image_path):
        """
        Add a known person to the database.
        
        Parameters:
            name       : the person's name (e.g., 'Alice')
            image_path : path to a clear photo of their face
        """
        img = face_recognition.load_image_file(image_path)
        encodings = face_recognition.face_encodings(img)
        
        if not encodings:
            print(f"  ⚠️  No face found in {image_path} — skipping.")
            return
        
        if name not in self.database:
            self.database[name] = []
        
        self.database[name].append(encodings[0])
        print(f"  ✅ Registered: {name} ({image_path})")
    
    def identify(self, image_path, top_k=1):
        """
        Identify the person(s) in an image.
        
        Returns a list of dicts, one per detected face:
            {'name': str, 'distance': float, 'location': (top, right, bottom, left)}
        """
        img = face_recognition.load_image_file(image_path)
        
        # Detect face locations
        locations = face_recognition.face_locations(img)
        if not locations:
            return [{'name': 'No face detected', 'distance': None, 'location': None}]
        
        # Compute embeddings for all detected faces
        query_encodings = face_recognition.face_encodings(img, locations)
        
        results = []
        
        for query_enc, location in zip(query_encodings, locations):
            best_name = 'Unknown'
            best_dist = float('inf')
            
            # Compare against every registered person
            for name, ref_encodings in self.database.items():
                # face_recognition.face_distance returns distances (lower = better)
                distances = face_recognition.face_distance(ref_encodings, query_enc)
                min_dist  = float(np.min(distances))
                
                if min_dist < best_dist:
                    best_dist = min_dist
                    best_name = name
            
            # Apply threshold: if too far away, call it Unknown
            if best_dist > self.threshold:
                best_name = 'Unknown'
            
            results.append({
                'name':     best_name,
                'distance': round(best_dist, 4),
                'location': location,   # (top, right, bottom, left)
                'match':    best_dist <= self.threshold
            })
        
        return results
    
    def summary(self):
        """Print a summary of the current database."""
        print("Face Recognition Database:")
        if not self.database:
            print("  (empty)")
        for name, encodings in self.database.items():
            print(f"  {name}: {len(encodings)} reference photo(s)")


print("✅ FaceRecognizer class defined!")

In [ ]:
# ── Run the Full Pipeline ────────────────────────────────────────────────────

# Step 1: Create recognizer
recognizer = FaceRecognizer(threshold=0.6)

# Step 2: Register known faces
print("Registering known faces...")
recognizer.register('Alice', 'faces/alice/alice_1.jpg')
recognizer.register('Alice', 'faces/alice/alice_2.jpg')
recognizer.register('Bob',   'faces/bob/bob_1.jpg')
recognizer.register('Bob',   'faces/bob/bob_2.jpg')

print()
recognizer.summary()

In [ ]:
# Step 3: Identify query faces
print("\nIdentifying query faces...\n")

for query_file, expected in [('faces/query/query_alice.jpg', 'Alice'),
                               ('faces/query/query_bob.jpg',   'Bob')]:
    results = recognizer.identify(query_file)
    
    print(f"Query: {query_file}  (expected: {expected})")
    for r in results:
        match_icon = '✅' if r['match'] else '❌'
        print(f"  {match_icon} Predicted: {r['name']}  |  Distance: {r['distance']}  "
              f"| Location: {r['location']}")
    print()

In [ ]:
# ── Visualise Recognition Results ────────────────────────────────────────────

def visualise_recognition(image_path, recognition_results):
    """
    Draw recognised identity labels on the image.
    """
    img = cv2.imread(image_path)
    
    for r in recognition_results:
        if r['location'] is None:
            continue
        
        top, right, bottom, left = r['location']
        name = r['name']
        dist = r['distance'] or 0
        is_match = r['match']
        
        # Colour: green = matched, red = unknown
        color = (0, 200, 0) if is_match else (0, 0, 200)
        
        # Draw bounding box
        cv2.rectangle(img, (left, top), (right, bottom), color, 2)
        
        # Background for label
        label = f"{name}  ({dist:.3f})"
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_DUPLEX, 0.6, 1)[0]
        cv2.rectangle(img, (left, bottom - label_size[1] - 10),
                       (left + label_size[0] + 6, bottom), color, -1)
        cv2.putText(img, label, (left + 3, bottom - 5),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    
    return img


# Visualise both queries
r_alice = recognizer.identify('faces/query/query_alice.jpg')
r_bob   = recognizer.identify('faces/query/query_bob.jpg')

vis_alice = visualise_recognition('faces/query/query_alice.jpg', r_alice)
vis_bob   = visualise_recognition('faces/query/query_bob.jpg', r_bob)

show_images_row([vis_alice, vis_bob],
                ['Query: Alice face', 'Query: Bob face'])

---
## 9. DeepFace — High-Level Library

**DeepFace** is a meta-library that wraps multiple state-of-the-art models behind a single, clean API. It supports:

| Model | Accuracy | Notes |
|---|---|---|
| `VGG-Face` | Good | Classic baseline |
| `Facenet` | Very good | Google |
| `Facenet512` | Very good | 512-dim |
| `ArcFace` | Excellent | SOTA |
| `DeepFace` | Good | Facebook |
| `SFace` | Excellent | Lightweight SOTA |

It also does **face analysis** in one call: age, gender, emotion, race.

In [ ]:
# ── DeepFace: Face Verification ───────────────────────────────────────────────

from deepface import DeepFace

# Verification: "Are these two images the SAME person?"
# Returns: {verified: True/False, distance, threshold, model, ...}

print("DeepFace Verification:\n")

# Alice vs Alice (should verify = True)
try:
    result_same = DeepFace.verify(
        img1_path='faces/alice/alice_1.jpg',
        img2_path='faces/alice/alice_2.jpg',
        model_name='ArcFace',          # Use ArcFace embeddings
        detector_backend='mtcnn',      # Use MTCNN for detection
        enforce_detection=False,       # Don't crash if detection fails
        silent=True
    )
    print(f"Alice ↔ Alice: verified={result_same['verified']}  distance={result_same['distance']:.4f}")
except Exception as e:
    print(f"Alice ↔ Alice: {e}")

# Alice vs Bob (should verify = False)
try:
    result_diff = DeepFace.verify(
        img1_path='faces/alice/alice_1.jpg',
        img2_path='faces/bob/bob_1.jpg',
        model_name='ArcFace',
        detector_backend='mtcnn',
        enforce_detection=False,
        silent=True
    )
    print(f"Alice ↔ Bob:   verified={result_diff['verified']}  distance={result_diff['distance']:.4f}")
except Exception as e:
    print(f"Alice ↔ Bob: {e}")

In [ ]:
# ── DeepFace: Face Analysis (Age, Gender, Emotion, Race) ──────────────────────

print("DeepFace Analysis (age, gender, emotion, race):\n")

try:
    analysis = DeepFace.analyze(
        img_path='faces/alice/alice_1.jpg',
        actions=['age', 'gender', 'emotion'],  # what to analyse
        detector_backend='mtcnn',
        enforce_detection=False,
        silent=True
    )
    
    # analysis is a list (one entry per face)
    for i, face_analysis in enumerate(analysis):
        print(f"Face {i+1}:")
        print(f"  Age:     {face_analysis.get('age', 'N/A')}")
        print(f"  Gender:  {face_analysis.get('dominant_gender', 'N/A')}")
        print(f"  Emotion: {face_analysis.get('dominant_emotion', 'N/A')}")
except Exception as e:
    print(f"Note: Analysis on synthetic image may fail: {e}")
    print("(This works well on real face photos.)")

In [ ]:
# ── DeepFace: Find — Search Through a Database of Images ───────────────────────

# DeepFace.find() searches a folder of images and returns the most similar faces

print("DeepFace Find (database search):\n")

try:
    # Search for the query face in the known faces folder
    df_results = DeepFace.find(
        img_path='faces/query/query_alice.jpg',
        db_path='faces/',               # folder containing subfolders of known faces
        model_name='ArcFace',
        detector_backend='mtcnn',
        enforce_detection=False,
        silent=True
    )
    
    print("Top matches:")
    if isinstance(df_results, list) and len(df_results) > 0:
        df = df_results[0]
        print(df[['identity', 'distance']].head(5).to_string(index=False))
    else:
        print(df_results)
        
except Exception as e:
    print(f"Note: DeepFace.find on synthetic images: {e}")
    print("(Works well with real face image folders.)")

---
## 10. Real-Time Face Recognition (Webcam)

The code below implements a real-time face recognition system using your webcam.

> Run this in a standalone `.py` file or terminal for best performance.
> Press **`q`** to quit.

In [ ]:
# ── Real-Time Face Recognition — Webcam ──────────────────────────────────────

def run_face_recognition_webcam(recognizer, process_every_n_frames=5):
    """
    Real-time face recognition using webcam.
    
    Parameters:
        recognizer           : FaceRecognizer with registered faces
        process_every_n_frames : only run recognition every N frames (for speed)
    """
    cap = cv2.VideoCapture(0)
    
    frame_count = 0
    last_results = []
    
    print("Webcam started. Press 'q' to quit.")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Only run recognition every N frames to maintain smooth FPS
        if frame_count % process_every_n_frames == 0:
            # Save frame temporarily and run recognition
            cv2.imwrite('_temp_frame.jpg', frame)
            last_results = recognizer.identify('_temp_frame.jpg')
        
        # Always draw the last known results
        for r in last_results:
            if r['location'] is None:
                continue
            
            top, right, bottom, left = r['location']
            name  = r['name']
            dist  = r['distance'] or 0
            color = (0, 200, 0) if r['match'] else (0, 0, 200)
            
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.putText(frame, f"{name} ({dist:.3f})",
                        (left, top - 8), cv2.FONT_HERSHEY_DUPLEX, 0.6,
                        color, 1, cv2.LINE_AA)
        
        cv2.putText(frame, f"Frame: {frame_count}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        
        cv2.imshow('Face Recognition — Press Q to quit', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    # Clean up temp file
    if os.path.exists('_temp_frame.jpg'):
        os.remove('_temp_frame.jpg')


# Uncomment to run:
# run_face_recognition_webcam(recognizer, process_every_n_frames=5)

print("ℹ️ Webcam function defined. Uncomment the last line to run it.")
print("   First, make sure you've registered your own face using recognizer.register()")

---
## 11. Performance & Accuracy Tips

### Registration best practices:
- Register **multiple photos** per person (different lighting, angles, expressions)
- Use **clear, well-lit** frontal face images
- Avoid sunglasses, heavy makeup, or occlusions

### Choosing the right threshold:
- Lower threshold = **stricter** (more "Unknown" results, fewer false positives)
- Higher threshold = **looser** (more matches, more false positives)
- Typical range: **0.4 – 0.7** for Euclidean distance

### Scaling the database:
- For **small databases** (< 100 people): linear search is fast enough
- For **large databases** (> 1000 people): use a **vector database** (FAISS, Chroma, Pinecone) for efficient nearest-neighbour search

In [ ]:
# ── Threshold Sensitivity Demo ────────────────────────────────────────────────
# Show how the threshold affects false accept rate vs false reject rate

# Simulate distances for same-person pairs (should be low)
np.random.seed(0)
same_person_distances = np.random.normal(loc=0.35, scale=0.08, size=200)
same_person_distances = np.clip(same_person_distances, 0, 1)

# Simulate distances for different-person pairs (should be high)
diff_person_distances = np.random.normal(loc=0.70, scale=0.12, size=200)
diff_person_distances = np.clip(diff_person_distances, 0, 1)

# Plot distributions
plt.figure(figsize=(10, 5))
plt.hist(same_person_distances, bins=30, alpha=0.6, color='green', label='Same person')
plt.hist(diff_person_distances, bins=30, alpha=0.6, color='red',   label='Different person')

# Show threshold
threshold = 0.55
plt.axvline(x=threshold, color='black', linestyle='--', linewidth=2,
            label=f'Threshold = {threshold}')

# Shade false regions
plt.axvspan(threshold, 1.0, alpha=0.05, color='green',
            label='False Rejects (same person, above threshold)')
plt.axvspan(0, threshold, alpha=0.05, color='red',
            label='False Accepts (diff person, below threshold)')

plt.xlabel('Euclidean Distance', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Face Distance Distribution — Threshold Effect', fontsize=13, fontweight='bold')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("The goal: pick a threshold in the 'gap' between the two distributions.")
print("Lower threshold → fewer false accepts, more false rejects.")
print("Higher threshold → fewer false rejects, more false accepts.")

---
## 12. Exercises 🎭

1. **Register yourself** — take 3 photos of yourself with your webcam (different angles), register them in the `FaceRecognizer`, then run identification on a new webcam photo.

2. **Threshold experiment** — Register Alice and Bob with real images. Try thresholds 0.4, 0.5, 0.6, 0.7. Print the prediction for each and observe how it changes.

3. **Multi-face image** — Find an image with 2+ people. Use `face_recognition.face_locations()` to detect all faces, then run identification on each.

4. **Alignment comparison** — Take a face photo and deliberately rotate it 20°. Run identification on both the original and rotated versions. Does alignment help?

5. **DeepFace analysis** — Run `DeepFace.analyze()` on a real photo and print the predicted age, gender, and emotion. How accurate is it?

---
## Summary

| Stage | Purpose | Tool Used |
|---|---|---|
| Detection | Find *where* faces are | MTCNN, face_recognition |
| Alignment | Normalise face orientation | eye landmarks + rotation |
| Embedding | Turn face into 128/512-dim vector | dlib ResNet / ArcFace |
| Matching | Compare vectors, find closest identity | Euclidean / cosine distance |
| Threshold | Decide same vs unknown | 0.4–0.7 (tune per use-case) |

### Key concepts:
- Face recognition is a **pipeline** — each stage matters
- **Alignment** is critical for robust recognition
- **ArcFace** embeddings generalise to **unseen identities**
- **Threshold** balances false accepts vs false rejects
- **DeepFace** provides a convenient high-level API over multiple SOTA models

### Next Steps
- Session 14+: Object Detection + Tracking pipeline (YOLOv8 + ByteTrack)

---
*Session 13B | Face Recognition Pipeline | Computer Vision & AI Course*